# Phase 2 — EfficientNet-B3 Submission (RSNA Knee Abnormality Detection)

Inference notebook for the `KneeEffNetMIL` (EfficientNet-B3 + per-finding GatedAttentionMIL) checkpoint `effnet_best.pt` (EMA weights, epoch 14, ema-macro 0.841).

**Run order on Kaggle:** attach (1) competition data, (2) a dataset containing `effnet_best.pt`. Set `PROBE=True` first (3 studies), then `PROBE=False` for the full run.

Single GPU only — DataParallel crashes T4 CUDA kernels (proven twice).

In [ ]:
# ================= CONFIG =================
VERSION = "phase2-effnet-submit-v1 (2026-10-07: EfficientNet-B3 single-GPU inference)"
print(f"VERSION: {VERSION}")

import os
from pathlib import Path

PROBE = True   # True: 3 studies to validate pipeline. False: full test run.

# --- checkpoint search (attach effnet_best.pt as a Kaggle dataset) ---
# Recursive search: find effnet_best.pt anywhere under /kaggle/input
CKPT_CANDIDATES = []
for _root, _dirs, _files in os.walk("/kaggle/input"):
    if "effnet_best.pt" in _files:
        CKPT_CANDIDATES.append(os.path.join(_root, "effnet_best.pt"))
# fallback: same-session training output dir
CKPT_CANDIDATES.append("/kaggle/working/effnet_ckpt/effnet_best.pt")
print(f"  ckpt search found: {CKPT_CANDIDATES}")
print(f"  /kaggle/input contents: {sorted(os.listdir('/kaggle/input'))}")
CKPT_PATH = next((p for p in CKPT_CANDIDATES if os.path.exists(p)), None)

# --- competition data ---
# Kaggle mounts competition data at /kaggle/input/competitions/<slug>/ (NOT /kaggle/input/<slug>/).
COMP_DATA = None
for _cand in ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
              "/kaggle/input/rsna-knee-abnormality-detection"]:
    if os.path.isdir(_cand):
        COMP_DATA = _cand
        break
print(f"  competition data dir: {COMP_DATA}")
if COMP_DATA:
    print(f"  contents: {sorted(os.listdir(COMP_DATA))}")
# Test images live in test_series/ (NOT test/).
TEST_DIR_CANDIDATES = ([f"{COMP_DATA}/test_series",
                        f"{COMP_DATA}/test",
                        f"{COMP_DATA}/test_images"] if COMP_DATA else [])
print(f"  test dir candidates: {TEST_DIR_CANDIDATES}")

# --- model (must match training notebook cell 7 exactly) ---
BACKBONE_NAME = "efficientnet_b3"   # what effnet_best.pt was trained with
IMAGE_SIZE = 288
N_SLICES = 8
PLANES = ["axial", "coronal", "sagittal"]
LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
          'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
N_FINDINGS = 12
ATTN_DIM = 128
SEED = 42

# --- inference ---
# Single-study forward passes: DICOM decode dominates runtime, and per-study
# isolation means one bad study can't poison a batch.
INFER_BATCH = 1         # studies per forward pass (fixed at 1 for robustness)
SAVE_EVERY = 25         # incremental submission.csv write interval
OUT_CSV = "/kaggle/working/submission.csv"

# --- phase-0 preprocessing (must match training cache) ---
P0_TARGET_PX = 336
P0_CROP_MM = 130.0
P0_SPACING_MM = P0_CROP_MM / P0_TARGET_PX   # ~= 0.3869 mm/px
P0_PCT_LO, P0_PCT_HI = 1.0, 99.0

print("CONFIG DUMP")
for k in ["BACKBONE_NAME","IMAGE_SIZE","N_SLICES","INFER_BATCH","OUT_CSV","PROBE"]:
    print(f"  {k} = {eval(k)}")
print(f"  CKPT_PATH = {CKPT_PATH}")
assert CKPT_PATH, "FATAL: effnet_best.pt not found — attach it as a Kaggle dataset"


In [ ]:
# ================= SETUP =================
import time, math, re, hashlib, warnings
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from scipy.ndimage import zoom as _zoom

t_start = time.time()
torch.manual_seed(SEED); np.random.seed(SEED)

# Single GPU only — DataParallel crashes CUDA kernels on T4 (proven twice).
assert torch.cuda.is_available(), "FATAL: no GPU"
DEVICE = torch.device("cuda:0")
torch.cuda.set_device(0)
print(f"torch {torch.__version__} | device: {torch.cuda.get_device_name(0)}")

# ---- discover test studies ----
test_dir = next((d for d in TEST_DIR_CANDIDATES if os.path.isdir(d)), None)
print(f"test dir: {test_dir}")
assert test_dir, f"FATAL: no test dir found in {TEST_DIR_CANDIDATES}"
_td_contents = sorted(os.listdir(test_dir))
print(f"  test_dir has {len(_td_contents)} entries")
print(f"  first 5 entries: {_td_contents[:5]}")

# sample submission defines the expected columns + study list
sample_sub = None
if COMP_DATA:
    for cand in [f"{COMP_DATA}/sample_submission.csv",
                 f"{COMP_DATA}/sampleSubmission.csv",
                 f"{COMP_DATA}/sample-submission.csv"]:
        if os.path.exists(cand):
            sample_sub = pd.read_csv(cand)
            print(f"  using sample submission: {cand}")
            break
if sample_sub is not None:
    print(f"sample_submission.csv: {sample_sub.shape}")
    print(f"  columns: {list(sample_sub.columns)}")
    print(f"  first 3 UIDs: {sample_sub['StudyInstanceUID'].tolist()[:3]}")
    test_uids = sample_sub["StudyInstanceUID"].tolist()
    missing = [u for u in test_uids if not os.path.isdir(os.path.join(test_dir, u))]
    print(f"  studies: {len(test_uids)} | missing dirs: {len(missing)}")
    if missing:
        print(f"  WARNING: {len(missing)} uids have no directory; they will get 0.5")
else:
    test_uids = sorted([d.name for d in Path(test_dir).iterdir() if d.is_dir()])
    print(f"no sample_submission.csv; discovered {len(test_uids)} study dirs")

SUB_COLS = (["StudyInstanceUID"] + list(sample_sub.columns[1:]) if sample_sub is not None
            else ["StudyInstanceUID"] + LABELS)
print(f"submission columns: {SUB_COLS}")

# PROBE: 3 studies only
if PROBE:
    test_uids = test_uids[:3]
    print(f"PROBE MODE: running on {len(test_uids)} studies only")

print(f"setup done in {time.time()-t_start:.1f}s")


In [ ]:
# ================= PHASE-0 LITE (test DICOM -> 336px volumes) =================
# Same deterministic pipeline as training cache. In-memory, no sharding.
# Uses scipy for resampling (no cv2 dependency).

FATSAT_RE = re.compile(r"fat[\s\-_]*sat")

def decode_slice(path):
    ds = pydicom.dcmread(str(path))
    x = ds.pixel_array.astype(np.float32)
    x = x * float(getattr(ds, "RescaleSlope", 1.0)) + float(getattr(ds, "RescaleIntercept", 0.0))
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        x = x.max() - x
    return x, ds

def geometric_order(datasets):
    keyed = []
    for ds in datasets:
        try:
            iop = [float(v) for v in ds.ImageOrientationPatient]
            ipp = [float(v) for v in ds.ImagePositionPatient]
            if len(iop) != 6 or len(ipp) != 3:
                raise ValueError("bad IOP/IPP")
            row = np.array(iop[0:3]); col = np.array(iop[3:6])
            normal = np.cross(row, col)
            n = float(np.linalg.norm(normal))
            if n < 1e-6:
                raise ValueError("degenerate IOP")
            normal = normal / n
            keyed.append((float(np.dot(np.array(ipp), normal)), ds))
        except Exception:
            keyed = None
            break
    if keyed is None:
        def _inum(ds):
            try: return int(ds.InstanceNumber)
            except Exception: return 0
        ordered = sorted(datasets, key=lambda ds: (_inum(ds), str(getattr(ds, "SOPInstanceUID", ""))))
        return ordered, None, True
    keyed.sort(key=lambda kv: kv[0])
    ds0 = keyed[0][1]
    iop = [float(v) for v in ds0.ImageOrientationPatient]
    normal = np.cross(np.array(iop[0:3]), np.array(iop[3:6]))
    normal = normal / float(np.linalg.norm(normal))
    return [ds for _, ds in keyed], normal, False

def window_series(vol, pct_lo=1.0, pct_hi=99.0):
    lo, hi = np.percentile(vol, [pct_lo, pct_hi])
    if not (np.isfinite(lo) and np.isfinite(hi)) or hi <= lo:
        return np.zeros_like(vol), False
    vol = np.clip(vol, lo, hi)
    return (vol - lo) / (hi - lo), True

def resample_slice(img, orig_spacing, target_spacing):
    sh, sw = img.shape
    nh = max(1, int(round(sh * orig_spacing[0] / target_spacing)))
    nw = max(1, int(round(sw * orig_spacing[1] / target_spacing)))
    if (nh, nw) == (sh, sw):
        return img
    return _zoom(img, (nh / sh, nw / sw), order=1).astype(np.float32)

def center_crop_pad(img, out_px, pad_value=0):
    h, w = img.shape
    ph = max(0, out_px - h); pw = max(0, out_px - w)
    if ph or pw:
        img = np.pad(img, ((ph // 2, ph - ph // 2), (pw // 2, pw - pw // 2)),
                     mode="constant", constant_values=pad_value)
        h, w = img.shape
    sh = (h - out_px) // 2; sw = (w - out_px) // 2
    return img[sh:sh + out_px, sw:sw + out_px]

def plane_from_iop(iop):
    row = np.array(iop[0:3]); col = np.array(iop[3:6])
    normal = np.cross(row, col)
    n = float(np.linalg.norm(normal))
    if n < 1e-6:
        return "unknown"
    normal = normal / n
    ax = int(np.argmax(np.abs(normal)))
    if abs(normal[ax]) < 0.8:
        return "unknown"
    return ["sagittal", "coronal", "axial"][ax]

def is_fluid_sensitive(ds):
    desc = str(getattr(ds, "SeriesDescription", "") or "")
    seq = str(getattr(ds, "ScanningSequence", "") or "")
    try: te = float(getattr(ds, "EchoTime", 0) or 0)
    except Exception: te = 0.0
    try: tr = float(getattr(ds, "RepetitionTime", 0) or 0)
    except Exception: tr = 0.0
    dl = desc.lower()
    toks = set(t for t in re.split(r"[^a-z0-9]+", dl) if t)
    seq_vals = [s.strip().upper() for s in seq.split("\\") if s.strip()]
    if "GR" in seq_vals:
        return False, "seq:GRE"
    fs_hit = sorted(toks & {"fs", "stir", "pd", "t2"})
    if FATSAT_RE.search(dl):
        fs_hit.append("fatsat")
    fs_hit.sort()
    if fs_hit:
        return True, "desc:" + "+".join(fs_hit)
    if "t1" in toks:
        return False, "desc:t1"
    if te >= 40:
        return True, "trte:TE>=40"
    if 0 < te < 20 and 0 < tr < 800:
        return False, "trte:T1-like"
    return False, "unknown->not-FS"

def canonicalize_laterality(vol, iop, laterality, normal):
    """Mirror R knees to canonical left; canonicalize sagittal stack order.
    (Full Phase-0 logic — training cache was canonicalized, so inference must be too.)"""
    X = np.array([1.0, 0.0, 0.0])
    if normal is not None:
        nx = abs(float(np.dot(normal, X)))
    else:
        nx = 0.0
    if nx > 0.8:
        # sagittal: force lateral->medial stack order for both L and R
        # lateral->medial runs along -X for L, +X for R
        if laterality == "R":
            return vol[::-1], True, "sagittal R: reversed to lateral->medial"
        return vol, False, "sagittal L/U: kept"
    # coronal/axial: mirror R only
    if laterality == "R":
        return vol[:, :, ::-1], True, "R mirrored to canonical left"
    return vol, False, "L/U: no-op"

def process_series(series_dir):
    files = sorted([f for f in Path(series_dir).iterdir()
                    if f.is_file() and f.suffix.lower() == ".dcm"])
    if not files:
        raise ValueError("no DICOM files")
    pairs = []
    for f in files:
        x, ds = decode_slice(f)
        pairs.append((ds, x))
    dss = [ds for ds, _ in pairs]
    ordered, normal, _fb = geometric_order(dss)
    arr_by_id = {id(ds): x for ds, x in pairs}
    vol = np.stack([arr_by_id[id(ds)] for ds in ordered]).astype(np.float32)
    if not np.all(np.isfinite(vol)):
        raise ValueError("non-finite pixels")
    vol, win_ok = window_series(vol, P0_PCT_LO, P0_PCT_HI)
    ds0 = ordered[0]
    try:
        ps = ds0.PixelSpacing
        spacing = (float(ps[0]), float(ps[1]))
    except Exception:
        raise ValueError("PixelSpacing missing")
    out = np.stack([resample_slice(s, spacing, P0_SPACING_MM) for s in vol])
    out = np.stack([center_crop_pad(s, P0_TARGET_PX, 0) for s in out])
    assert out.shape[1] == P0_TARGET_PX and out.shape[2] == P0_TARGET_PX
    try:
        iop = [float(v) for v in ds0.ImageOrientationPatient]
    except Exception:
        raise ValueError("ImageOrientationPatient missing")
    plane_tag = str(getattr(ds0, "Anatomical_Plane", "") or "").strip().lower()
    plane = plane_tag if plane_tag in ("sagittal", "coronal", "axial") else plane_from_iop(iop)
    fs, fs_reason = is_fluid_sensitive(ds0)
    meta = {
        "series_uid": str(getattr(ds0, "SeriesInstanceUID", Path(series_dir).name)),
        "plane": plane, "is_fs": bool(fs), "fs_reason": fs_reason,
        "n_slices": int(out.shape[0]),
        "laterality": str(getattr(ds0, "ImageLaterality", "") or "").strip().upper(),
        "iop": iop, "normal": normal,
    }
    return out, meta

def select_series(series_infos):
    by_plane_fs, by_plane_all = {}, {}
    for vol, meta in series_infos:
        if meta["plane"] not in ("sagittal", "coronal", "axial"):
            continue
        by_plane_all.setdefault(meta["plane"], []).append((vol, meta))
        if meta["is_fs"]:
            by_plane_fs.setdefault(meta["plane"], []).append((vol, meta))
    selected = {}
    for plane in ("sagittal", "coronal", "axial"):
        cands = by_plane_fs.get(plane, [])
        if not cands:
            cands = by_plane_all.get(plane, [])
        if not cands:
            continue
        cands.sort(key=lambda vm: (-vm[1]["n_slices"], vm[1]["series_uid"]))
        selected[plane] = cands[0]
    return selected

def preprocess_study(study_dir):
    _V = True  # verbose per-study preprocessing
    if _V: print(f"    [preprocess] dir: {study_dir}")
    """Test DICOM study -> {plane: uint8 (S,336,336)}, plane_mask. Never crashes caller."""
    series_dirs = sorted([d for d in Path(study_dir).iterdir() if d.is_dir()])
    if not series_dirs:
        raise ValueError("no series directories")
    infos = []
    for sdir in series_dirs:
        try:
            infos.append(process_series(sdir))
        except Exception:
            continue
    if not infos:
        raise ValueError("all series failed")
    lats = [m["laterality"] for _, m in infos if m["laterality"] in ("L", "R")]
    laterality = max(sorted(set(lats)), key=lats.count) if lats else "U"
    selected = select_series(infos)
    if not selected:
        raise ValueError("no usable plane")
    planes_out, mask = {}, []
    for p in PLANES:
        if p in selected:
            vol, meta = selected[p]
            normal = None if meta["normal"] is None else np.array(meta["normal"])
            vol_m, _, _ = canonicalize_laterality(vol, meta["iop"], laterality, normal)
            u8 = np.clip(vol_m, 0.0, 1.0)
            u8 = (u8 * 255.0 + 0.5).astype(np.uint8)
            planes_out[p] = u8
            mask.append(1.0)
        else:
            mask.append(0.0)
    return planes_out, mask

print("phase-0 lite: loaded")


In [ ]:
# ================= MODEL (single GPU, no DataParallel) =================
# KneeEffNetMIL copied from training notebook cell 7 — must match exactly
# for checkpoint loading.
import timm
assert BACKBONE_NAME in timm.list_models(), f"FATAL: {BACKBONE_NAME} not in timm"
print(f"backbone: {BACKBONE_NAME}")

class GatedAttentionMIL(nn.Module):
    def __init__(self, feat_dim, attn_dim=ATTN_DIM):
        super().__init__()
        self.V = nn.Linear(feat_dim, attn_dim)
        self.U = nn.Linear(feat_dim, attn_dim)
        self.w = nn.Linear(attn_dim, 1)
    def forward(self, h):  # h: (B, S, D)
        a = self.w(torch.tanh(self.V(h)) * torch.sigmoid(self.U(h)))
        a = F.softmax(a, dim=1)
        return (a * h).sum(dim=1), a.squeeze(-1)

class KneeEffNetMIL(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, attn_dim=ATTN_DIM,
                 n_findings=N_FINDINGS, pretrained=False,
                 drop_path_rate=0.1):
        super().__init__()
        # Inference: backbone weights come from the checkpoint, not timm download.
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, drop_path_rate=drop_path_rate)
        D = self.backbone.num_features
        self.mil_heads = nn.ModuleList([GatedAttentionMIL(D, attn_dim)
                                        for _ in range(n_findings)])
        self.heads = nn.ModuleList([nn.Linear(D, 1) for _ in range(n_findings)])
    def forward_plane(self, x):  # x: (B, S, 3, H, W)
        B, S = x.shape[:2]
        # reshape (not view): non-contiguous tensors crash view() on CUDA
        h = self.backbone(x.reshape(B * S, *x.shape[2:])).reshape(B, S, -1)
        return torch.stack([mil(h)[0] for mil in self.mil_heads], dim=1)  # (B,12,D)
    def forward(self, planes, plane_mask):
        feats = torch.stack([self.forward_plane(planes[p]) for p in PLANES], dim=1)
        m = plane_mask.reshape(-1, 3, 1, 1)
        fused = (feats * m).sum(1) / m.sum(1).clamp(min=1e-6)  # (B,12,D)
        return torch.cat([self.heads[f](fused[:, f, :])
                          for f in range(N_FINDINGS)], dim=1)  # (B,12)

model = KneeEffNetMIL().to(DEVICE)
# NO DataParallel — it corrupts the CUDA context on T4. Single GPU only.
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
print(f"checkpoint keys: {list(ckpt.keys())}")
# Prefer EMA weights (best ema-macro 0.841 @ ep14); fall back to raw model.
state_key = "ema" if "ema" in ckpt else "model"
tgt = model.module if isinstance(model, nn.DataParallel) else model
_incompat = tgt.load_state_dict(ckpt[state_key], strict=False)
print(f"loaded {CKPT_PATH} [{state_key}] "
      f"(epoch {ckpt.get('epoch')}, best {ckpt.get('best')})")
print(f"  load_state_dict: missing={_incompat.missing_keys}, unexpected={_incompat.unexpected_keys}")
assert not _incompat.missing_keys and not _incompat.unexpected_keys, \
    "FATAL: state dict mismatch — model definition drifted from training"
model.eval()
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"params: {n_params:.1f}M | model on {DEVICE}, eval mode")

# smoke test: random input forward
with torch.no_grad():
    _pb = {p: torch.randn(1, N_SLICES, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE) for p in PLANES}
    _o = model(_pb, torch.ones(1, 3).to(DEVICE))
    assert _o.shape == (1, N_FINDINGS), f"bad output shape {tuple(_o.shape)}"
    _pr = torch.sigmoid(_o)
    assert torch.all(torch.isfinite(_pr)) and _pr.min() >= 0 and _pr.max() <= 1
print("model smoke test: ALL GREEN")


In [ ]:
# ================= INFERENCE =================
def sample_slices(vol, n):
    S = vol.shape[0]
    if S == n:
        return vol
    idx = np.linspace(0, S - 1, n).round().astype(int)
    return vol[idx]

def study_to_batch(planes_u8, plane_mask):
    """{plane: (S,336,336) uint8} -> model-ready tensors (no augmentation)."""
    planes = {}
    for p, m in zip(PLANES, plane_mask):
        if m == 1.0:
            vol = sample_slices(planes_u8[p], N_SLICES).astype(np.float32) / 255.0
            t = torch.from_numpy(vol).unsqueeze(1)  # (S,1,336,336)
            t = F.interpolate(t, size=(IMAGE_SIZE, IMAGE_SIZE),
                              mode="bilinear", align_corners=False)
            vol = t.squeeze(1).numpy().astype(np.float32)
        else:
            vol = np.zeros((N_SLICES, IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
        planes[p] = np.repeat(vol[:, None, :, :], 3, axis=1).astype(np.float32)
    return planes, np.array(plane_mask, dtype=np.float32)

def _write_csv(results):
    rows = [[uid] + [float(p) for p in probs] for uid, probs in results]
    df = pd.DataFrame(rows, columns=SUB_COLS)
    df.to_csv(OUT_CSV, index=False)

@torch.no_grad()
def predict_studies(uids, test_dir):
    """Run inference. Returns list of (uid, probs12). Failed study -> 0.5s."""
    results, failed = [], []
    t0 = time.time()
    for i, uid in enumerate(uids):
        _verbose_study = (i < 3)  # detailed logging for first 3 studies
        try:
            sdir = os.path.join(test_dir, uid)
            if _verbose_study:
                print(f"  [study {i+1}] uid={uid} | sdir exists: {os.path.isdir(sdir)}")
            if not os.path.isdir(sdir):
                raise ValueError("study dir missing")
            planes_u8, pmask = preprocess_study(sdir)
            if _verbose_study:
                for _p in PLANES:
                    _v = planes_u8.get(_p)
                    print(f"    plane {_p}: shape={_v.shape if _v is not None else None}, "
                          f"mask={pmask[PLANES.index(_p)]}")
            planes_np, pmask_np = study_to_batch(planes_u8, pmask)
            batch = {p: torch.from_numpy(planes_np[p]).unsqueeze(0).to(DEVICE) for p in PLANES}
            logits = model(batch, torch.from_numpy(pmask_np).unsqueeze(0).to(DEVICE))
            probs = torch.sigmoid(logits).double().cpu().numpy()[0]  # float64 end-to-end
            assert probs.shape == (N_FINDINGS,) and np.all(np.isfinite(probs))
            if _verbose_study:
                print(f"    probs: {[f'{v:.3f}' for v in probs]}")
        except Exception as e:
            probs = np.full(N_FINDINGS, 0.5)
            failed.append(uid)
            if len(failed) <= 5:
                import traceback
                print(f"  FAILED {uid}: {type(e).__name__}: {e}")
                traceback.print_exc()
        results.append((uid, probs))
        # incremental save + progress
        if (i + 1) % SAVE_EVERY == 0 or (i + 1) == len(uids):
            _write_csv(results)
            dt = time.time() - t0
            rate = (i + 1) / dt
            eta = (len(uids) - i - 1) / rate if rate > 0 else 0
            print(f"  {i+1}/{len(uids)} | {dt:.0f}s elapsed | {eta/60:.0f}min ETA | "
                  f"failed: {len(failed)}", flush=True)
    return results, failed

print(f"running inference on {len(test_uids)} studies (PROBE={PROBE})...")
if not PROBE and len(test_uids) < 100:
    print(f"  WARNING: only {len(test_uids)} studies found! Expected ~1300.")
results, failed = predict_studies(test_uids, test_dir)
_write_csv(results)  # final write
print(f"\ndone: {len(results)} studies, {len(failed)} failed (0.5-filled)")
if failed:
    print(f"  failed uids (first 10): {failed[:10]}")
print(f"submission.csv -> {OUT_CSV}")
print(f"total inference time: {(time.time()-t_start)/60:.1f} min")


In [ ]:
# ================= VERIFY SUBMISSION =================
sub = pd.read_csv(OUT_CSV)
print(f"shape: {sub.shape} (expect ({len(test_uids)}, 13))")
assert list(sub.columns) == SUB_COLS, f"column mismatch: {list(sub.columns)}"
assert len(sub) == len(test_uids), "row count mismatch"
assert sub["StudyInstanceUID"].tolist() == test_uids, "uid order mismatch"
assert sub[LABELS].notna().all().all(), "NaNs in predictions"
assert ((sub[LABELS] >= 0) & (sub[LABELS] <= 1)).all().all(), "probs out of [0,1]"
print("no NaNs, all probs in [0,1], uids match")
print("\nhead:")
print(sub.head(3).round(4).to_string(index=False))
print("\nlabel means (sanity: should roughly match train priors):")
print(sub[LABELS].mean().round(3).to_string())
print("\nSUBMISSION VERIFIED — upload", OUT_CSV, "to Kaggle")
